## NOTEBOOK 02: ADVANCED PREPROCESSING, CYCLICAL ENCODING, AND STRATIFIED SPLITTING



In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

print("=== Loading Checkpoint Matrix from Notebook 01 ===")
# Ingest data state securely saved from your Notebook 1 run
df = pd.read_csv("processed_eda_checkpoint.csv")

=== Loading Checkpoint Matrix from Notebook 01 ===


In [ ]:
# ==========================================
# 1. DROP UNINFORMATIVE & POST-FACTO COLUMNS
# ==========================================
# Remove 100% missing columns, system PII strings, and tracking IDs to save RAM
dead_columns = [
    'description', 'closed_datetime', 'action_taken_timestamp',  # 100% missing columns
    'vehicle_number', 'updated_vehicle_number',                 # Raw PII identifiers
    'device_id', 'created_by_id', 'id',                         # Database keys
    'data_sent_to_scita_timestamp', 'validation_timestamp'      # Post-facto process metrics
]
df = df.drop(columns=[col for col in dead_columns if col in df.columns])

In [ ]:
# ==========================================
# 2. SPATIAL REPAIR & HIGH-CARDINALITY RATIOS
# ==========================================
print("Imputing spatial indices and mapping frequency encodings...")

# Impute minor missing coordinates using spatial medians to keep your 298k dataset intact
df['latitude'] = df['latitude'].fillna(df['latitude'].median())
df['longitude'] = df['longitude'].fillna(df['longitude'].median())

# Guard categorical columns from breaking downstream math operations
df['police_station'] = df['police_station'].fillna("UNKNOWN_STATION")
df['junction_name'] = df['junction_name'].fillna("UNKNOWN_JUNCTION")

# Frequency/density encoding maps categorical strings directly to their presence ratio in the data.
# This prevents high-cardinality blocks from consuming massive RAM like One-Hot Encoding would.
station_frequency_map = df['police_station'].value_counts(normalize=True).to_dict()
df['police_station_encoded'] = df['police_station'].map(station_frequency_map)

junction_frequency_map = df['junction_name'].value_counts(normalize=True).to_dict()
df['junction_encoded'] = df['junction_name'].map(junction_frequency_map)

Imputing spatial indices and mapping frequency encodings...


In [ ]:
# ==========================================
# 3. TRIGONOMETRIC CYCLICAL TIME TRANSFORMATION
# ==========================================
print("Converting chronological boundaries into continuous trigonometric vectors...")
# Transforming hours (0-23) and days (0-6) into sine/cosine loops ensures the
# classification model naturally captures that hour 23 sits directly adjacent to hour 0.
df['hour_sin'] = np.sin(2 * np.pi * df['hour_of_day'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour_of_day'] / 24.0)
df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7.0)
df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7.0)

Converting chronological boundaries into continuous trigonometric vectors...


In [ ]:
# ==========================================
# 4. MATRIX EXCLUSION & FEATURE SELECTION
# ==========================================
ENGINEERED_FEATURES = [
    'latitude', 'longitude',
    'vehicle_weight', 'violation_severity',
    'police_station_encoded', 'junction_encoded',
    'hour_sin', 'hour_cos',
    'day_sin', 'day_cos'
]
TARGET_LABEL = 'high_impact_bottleneck'

X = df[ENGINEERED_FEATURES]
y = df[TARGET_LABEL]

In [ ]:
# ==========================================
# 5. STRATIFIED SYSTEMIC PARTITION SPLITTING
# ==========================================
print("Partitioning records into fully isolated train-validation-test blocks (70/15/15)...")
# Stratifying is critical here: it ensures your 32.9% positive class distribution
# is maintained perfectly across your training, validation, and test arrays.
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f" -> Training Rows Generated:   {X_train.shape[0]:,}")
print(f" -> Validation Rows Generated: {X_val.shape[0]:,}")
print(f" -> Testing Rows Generated:    {X_test.shape[0]:,}")

Partitioning records into fully isolated train-validation-test blocks (70/15/15)...
 -> Training Rows Generated:   208,915
 -> Validation Rows Generated: 44,767
 -> Testing Rows Generated:    44,768


In [ ]:
# ==========================================
# 6. VARIABLE VARIANCE SCALING
# ==========================================
print("Normalizing feature magnitudes...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert arrays back into structured Pandas DataFrames to retain operational columns labels
X_train_final = pd.DataFrame(X_train_scaled, columns=ENGINEERED_FEATURES)
X_val_final = pd.DataFrame(X_val_scaled, columns=ENGINEERED_FEATURES)
X_test_final = pd.DataFrame(X_test_scaled, columns=ENGINEERED_FEATURES)

Normalizing feature magnitudes...


In [ ]:
# ==========================================
# 7. EXPORT PROCESSED ARRAYS TO WORKSPACE
# ==========================================
print("Writing structured array packages directly to disk...")
joblib.dump(scaler, "preprocessor_scaler.pkl")

X_train_final.to_csv("X_train_clean.csv", index=False)
X_val_final.to_csv("X_val_clean.csv", index=False)
X_test_final.to_csv("X_test_clean.csv", index=False)

y_train.to_csv("y_train.csv", index=False)
y_val.to_csv("y_val.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("\n[SUCCESS] Notebook 02 runtime phase executed safely. Downstream assets locked.")

Writing structured array packages directly to disk...

[SUCCESS] Notebook 02 runtime phase executed safely. Downstream assets locked.
